# BIOSTAT 707 — Checkpoint 1
## ICU cohort description and missingness

**Author:** Siyu Gong  
**NetID:** sg763

### 1. Setup

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown, Image
from tableone import TableOne

ROOT = Path.cwd()
assert (ROOT / "checkpoint1.ipynb").exists(), "Run from the project folder."
DATA, OUT = ROOT / "data", ROOT / "output"
OUT.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid", font_scale=0.9)

### 2. Check data completeness

In [ ]:
def check_set_a():
    try:
        files = sorted((DATA / "set-a").glob("*.txt"))
        assert len(files) == 4000, f"Expected 4,000 files, found {len(files):,}."
        assert (DATA / "Outcomes-a.txt").is_file(), "Missing Outcomes-a.txt."

        outcomes = pd.read_csv(DATA / "Outcomes-a.txt").set_index("RecordID")
        assert len(outcomes) == 4000, "Expected 4,000 outcome records."
        assert outcomes.index.is_unique, "Duplicate RecordIDs in outcomes."
        assert set(outcomes.index) == {int(f.stem) for f in files}, "RecordIDs do not match."
        assert outcomes["In-hospital_death"].isin([0, 1]).all(), "Invalid death values."

    except Exception as error:
        print(f"Data check failed: {error}")
        raise

    print(f"Data check passed: {len(files):,} ICU files and {len(outcomes):,} outcome records.")
    return files, outcomes

files, outcomes = check_set_a()

### 3. Load and save the tables

`load_set_a()` reads the ICU record files and returns:

- `long`: time-series measurements. Each row contains one measurement for one ICU stay at one recorded time. It contains `RecordID`, `Time`, `Parameter`, `ValueRaw`, and `Value`.
- `static`: admission descriptors in wide format. Each row represents one ICU stay, indexed by `RecordID`. Columns are age, gender, height, ICU type, and admission weight.

Admission weight comes from the time-zero record and is named `AdmissionWeight`. All weight records, including the admission record, also remain in `long` because weight can be measured repeatedly.

In [ ]:
def load_set_a():
    frames = []
    for file in files:
        frame = pd.read_csv(file)
        assert list(frame.columns) == ["Time", "Parameter", "Value"]
        assert frame.loc[frame.Parameter.eq("RecordID"), "Value"].tolist() == [int(file.stem)]
        frames.append(frame.assign(RecordID=int(file.stem)))
    raw = pd.concat(frames, ignore_index=True).rename(columns={"Value": "ValueRaw"})
    v = raw.ValueRaw
    other_negative = v.lt(0) & v.ne(-1)
    raw["Value"] = v.mask(v.lt(0))
    names = ["Age", "Gender", "Height", "ICUType", "Weight"]
    static = raw[raw.Time.eq("00:00") & raw.Parameter.isin(names)].pivot(
        index="RecordID", columns="Parameter", values="Value").rename(columns={"Weight": "AdmissionWeight"})
    long = raw.loc[~raw.Parameter.isin(["RecordID", "Age", "Gender", "Height", "ICUType"]),
                   ["RecordID", "Time", "Parameter", "ValueRaw", "Value"]].copy()
    print(f"{len(files):,} ICU stays; {len(raw):,} original rows; {len(long):,} time-series rows.")
    print(f"Missing sentinel: -1; sentinel rows: {v.eq(-1).sum():,}; other negative rows: {other_negative.sum():,}.")
    assert static.index.is_unique and set(static.index) == {int(f.stem) for f in files}
    ranges = raw.assign(Known=v.mask(v.eq(-1)), Missing=v.eq(-1)).groupby("Parameter").agg(
        Count=("ValueRaw", "size"), Raw_min=("ValueRaw", "min"), Raw_max=("ValueRaw", "max"),
        Missing_code_count=("Missing", "sum"), Min_without_minus1=("Known", "min"),
        Max_without_minus1=("Known", "max"))
    ranges.to_csv(OUT / "value_ranges.csv")
    return long, static

long, static = load_set_a()

long.to_csv(OUT / "set-a_long.csv", index=False)
static.to_csv(OUT / "static.csv")

In [ ]:
print(f"Long table: {len(long):,} measurement rows; {long.Parameter.nunique()} time-series variables.")
display(long.head())

In [ ]:
print(f"Descriptor table: {len(static):,} ICU stays; {len(static.columns)} admission variables; RecordID is the index.")
display(static.head())

Summarize the full observation window.

In [ ]:
time = long.Time.str.split(":", expand=True).astype(int)
minutes = time[0] * 60 + time[1]
assert time[1].between(0, 59).all() and minutes.between(0, 2880).all()
print("Time-series window checked: 0–48 hours, including both endpoints.")
variables = sorted(long.Parameter.unique())
measured = long.dropna(subset=["Value"])
measurement_minutes = minutes.loc[measured.index]
keys = ["RecordID", "Parameter"]
summary = measured.groupby(keys).Value.agg(["count", "min", "max", "mean"])
for stat, endpoint in [("first", "min"), ("last", "max")]:
    endpoint_time = measurement_minutes.groupby([measured.RecordID, measured.Parameter]).transform(endpoint)
    summary[stat] = measured.loc[measurement_minutes.eq(endpoint_time)].groupby(keys).Value.mean()
summary = summary[["count", "first", "last", "min", "max", "mean"]].unstack("Parameter")
summary.columns = [f"{variable}_{stat}" for stat, variable in summary.columns]
wide = outcomes.join(static).join(summary)
for variable in variables:
    wide[f"{variable}_count"] = wide[f"{variable}_count"].fillna(0).astype(int)
    wide[f"{variable}_missing"] = wide[f"{variable}_count"].eq(0).astype(int)
assert len(wide) == 4000 and wide.index.is_unique
wide.to_csv(OUT / "set-a_wide.csv")
predictors = wide.drop(columns=outcomes.columns)

In [ ]:
print(f"Wide table: {len(wide):,} stays; {len(predictors.columns):,} candidate predictors; outcomes excluded from predictors.")
display(wide.head())

### 4. Check the original value ranges

In [ ]:
ranges = pd.read_csv(OUT / "value_ranges.csv", index_col="Parameter")
display(ranges)

In [ ]:
height_missing = int(ranges.loc["Height", "Missing_code_count"])
weight_missing = int((long.Parameter.eq("Weight") & long.Time.eq("00:00") & long.ValueRaw.eq(-1)).sum())
gender_missing = int(ranges.loc["Gender", "Missing_code_count"])
missing_variables = sum(wide[f"{v}_missing"].any() for v in variables)
display(Markdown(
    f"- Unreasonable values: pH ranges from {ranges.loc['pH', 'Min_without_minus1']:g} to "
    f"{ranges.loc['pH', 'Max_without_minus1']:g}, height ranges from "
    f"{ranges.loc['Height', 'Min_without_minus1']:g} to {ranges.loc['Height', 'Max_without_minus1']:g} cm, "
    f"the lowest weight is {ranges.loc['Weight', 'Min_without_minus1']:g} kg, and the lowest temperature is "
    f"{ranges.loc['Temp', 'Min_without_minus1']:g}°C. These values may be recording errors. "
    "Zero values in heart rate, blood pressure, and PaO₂ are also unreasonable.\n\n"
    f"- Variables marked as missing (−1): Height has {height_missing:,} missing records, "
    f"accounting for {height_missing / len(static):.2%} of patients. Admission weight has "
    f"{weight_missing:,} missing records ({weight_missing / len(static):.2%}), and gender has "
    f"{gender_missing:,} missing records ({gender_missing / len(static):.3%}).\n\n"
    f"- Variables with no available measurements: {missing_variables} of the {len(variables)} "
    "time-series variables have at least one patient with no available measurements."))

### 5. What exists at prediction time

Prediction goal: At 48 hours after ICU admission, predict whether the patient will die during the hospital stay.

Information we can use as predictors:

- General Descriptors: age, gender, height, ICU type, and admission weight.
- Time Series variables from the first 48 hours: Albumin, ALP, ALT, AST, Bilirubin, BUN, Cholesterol, Creatinine, DiasABP, FiO2, GCS, Glucose, HCO3, HCT, HR, K, Lactate, Mg, MAP, MechVent, Na, NIDiasABP, NIMAP, NISysABP, PaCO2, PaO2, pH, Platelets, RespRate, SaO2, SysABP, Temp, TroponinI, TroponinT, Urine, WBC, and Weight.
- Missingness information in the first 48 hours.

Information we cannot use as predictors:

- Measurements after the first 48 hours.
- Final hospital length of stay.
- Final survival time.
- In-hospital death.

### 6. Table 1: characteristics by outcome

- All admission characteristics and time-series variables are included. Continuous values are mean [Q1, Q3] (mean [25th percentile, 75th percentile]) across stays; categorical values are n (%). Dynamic variables use each stay's mean, except minimum GCS. Admission weight is separate from mean weight.
- Values of -1 and other negative values are treated as missing. A variable is also missing for a patient if there are no available measurements during the entire 48-hour period.

In [ ]:
table_data = wide.copy()
table_data["Outcome"] = wide["In-hospital_death"].map({0: "Survived", 1: "Died"})
table_data["Sex"] = wide.Gender.map({0: "Female", 1: "Male"}).fillna("Unknown")
icu_names = {1: "Coronary care", 2: "Cardiac surgery recovery", 3: "Medical ICU", 4: "Surgical ICU"}
table_data["ICU"] = wide.ICUType.map(icu_names)
table_data["MechVent"] = wide.MechVent_max.map({1: "Yes (recorded)"}).fillna("Unknown")
clinical = [f"{v}_min" if v == "GCS" else f"{v}_mean" for v in variables if v != "MechVent"]
continuous = ["Age", "Height", "AdmissionWeight"] + clinical
categorical = ["Sex", "ICU", "MechVent"]
units = {
    "Albumin": "g/dL", "ALP": "IU/L", "ALT": "IU/L", "AST": "IU/L", "Bilirubin": "mg/dL",
    "BUN": "mg/dL", "Cholesterol": "mg/dL", "Creatinine": "mg/dL", "DiasABP": "mmHg",
    "FiO2": "fraction", "GCS": "score", "Glucose": "mg/dL", "HCO3": "mmol/L", "HCT": "%",
    "HR": "bpm", "K": "mEq/L", "Lactate": "mmol/L", "Mg": "mmol/L", "MAP": "mmHg",
    "Na": "mEq/L", "NIDiasABP": "mmHg", "NIMAP": "mmHg", "NISysABP": "mmHg",
    "PaCO2": "mmHg", "PaO2": "mmHg", "pH": "pH units", "Platelets": "cells/nL",
    "RespRate": "/min", "SaO2": "%", "SysABP": "mmHg", "Temp": "°C",
    "TroponinI": "µg/L", "TroponinT": "µg/L", "Urine": "mL", "WBC": "cells/nL", "Weight": "kg"}
labels = {"Age": "Age (years)", "Height": "Height (cm)", "AdmissionWeight": "Admission weight (kg)",
          "MechVent": "MechVent documentation"}
for column in clinical:
    variable, stat = column.rsplit("_", 1)
    labels[column] = f"{variable}, {stat} ({units[variable]})"
table1 = TableOne(table_data, columns=categorical + continuous, categorical=categorical,
    groupby="Outcome", label_suffix=False, rename=labels, pval=False,
    order={"Outcome": ["Survived", "Died"]}, decimals={"FiO2_mean": 2, "Creatinine_mean": 2, "pH_mean": 2})
table_summary = table1.tableone.copy()
groups = {"Overall": table_data, "Survived": table_data[table_data.Outcome.eq("Survived")],
          "Died": table_data[table_data.Outcome.eq("Died")]}
for column in continuous:
    digits = 2 if column in ["FiO2_mean", "Creatinine_mean", "pH_mean"] else 1
    for group, data in groups.items():
        values = data[column].dropna()
        mean = values.mean()
        q1, q3 = values.quantile([0.25, 0.75])
        table_summary.loc[(labels[column], ""), ("Grouped by Outcome", group)] = (
            f"{mean:.{digits}f} [{q1:.{digits}f}, {q3:.{digits}f}]")
table_summary = table_summary.rename(
    index={labels[column]: f"{labels[column]}, mean [Q1,Q3]" for column in continuous}, level=0)
table_display = table_summary.copy()
table_display.columns = table_display.columns.get_level_values(-1)
table_display.index.names = ["Characteristic", "Category"]
table_display = table_display.reset_index().fillna("")
table_display["Characteristic"] = table_display.Characteristic.str.replace(
    r", mean \[Q1,Q3\]", "", regex=True).replace({"n": "ICU stays, n"})
group_starts = table_display.Characteristic.ne(table_display.Characteristic.shift())
table_display.loc[~group_starts, "Characteristic"] = ""
table_display = table_display[["Characteristic", "Category", "Overall", "Survived", "Died", "Missing"]]
styles = {
    "": "width:100%; border-collapse:collapse; font-family:Arial,sans-serif; color:#243447",
    "caption": "text-align:left; font-size:17px; font-weight:600; padding:0 0 12px",
    "thead th": "background:#243f56; color:white; padding:12px; text-align:right",
    "thead th:nth-child(-n+2)": "text-align:left",
    "tbody tr:nth-child(even)": "background:#f3f6f9",
    "tbody tr:first-child": "background:#e4edf4; font-weight:600",
    "td": "border-bottom:1px solid #e5eaf0",
    "tbody tr:hover": "background:#eaf2f8"}
display(table_display.style.hide(axis="index").set_uuid("table1")
    .set_caption("Table 1 · Mean [Q1, Q3] or n (%) by in-hospital outcome")
    .set_properties(**{"font-size": "13px", "padding": "9px 12px", "text-align": "right",
                       "font-variant-numeric": "tabular-nums", "white-space": "nowrap"})
    .set_properties(subset=["Characteristic", "Category"], **{"text-align": "left", "white-space": "normal"})
    .set_properties(subset=["Missing"], **{"color": "#64748b"})
    .set_table_styles([{"selector": selector, "props": css} for selector, css in styles.items()]))
table_summary.to_csv(OUT / "table1.csv")

In [ ]:
outcome_counts = table_data.Outcome.value_counts()
overall_mortality = wide["In-hospital_death"].mean()
group_means = table_data.groupby("Outcome")[[
    "Age", "Lactate_mean", "BUN_mean", "Creatinine_mean", "Albumin_mean", "GCS_min"]].mean()
group_percentages = table_data.assign(
    MedicalICU=table_data.ICU.eq("Medical ICU"),
    RecordedMechVent=table_data.MechVent.eq("Yes (recorded)")
).groupby("Outcome")[["MedicalICU", "RecordedMechVent"]].mean() * 100
display(Markdown(
    f"- Table 1 includes {len(wide):,} ICU stays. There were {outcome_counts['Survived']:,} survivors "
    f"and {outcome_counts['Died']:,} deaths in hospital. The death rate was {overall_mortality:.2%}. "
    "The death group had a higher mean age than the survivor group "
    f"({group_means.loc['Died', 'Age']:.1f} vs {group_means.loc['Survived', 'Age']:.1f} years). "
    "The death group also had a higher percentage of medical ICU patients "
    f"({group_percentages.loc['Died', 'MedicalICU']:.1f}% vs {group_percentages.loc['Survived', 'MedicalICU']:.1f}%) "
    "and patients with mechanical ventilation records "
    f"({group_percentages.loc['Died', 'RecordedMechVent']:.1f}% vs {group_percentages.loc['Survived', 'RecordedMechVent']:.1f}%).\n\n"
    "- For patients with available measurements, the death group had higher mean lactate "
    f"({group_means.loc['Died', 'Lactate_mean']:.1f} vs {group_means.loc['Survived', 'Lactate_mean']:.1f} mmol/L), "
    f"BUN ({group_means.loc['Died', 'BUN_mean']:.1f} vs {group_means.loc['Survived', 'BUN_mean']:.1f} mg/dL), "
    f"and creatinine ({group_means.loc['Died', 'Creatinine_mean']:.2f} vs {group_means.loc['Survived', 'Creatinine_mean']:.2f} mg/dL). "
    "Mean albumin was lower in the death group "
    f"({group_means.loc['Died', 'Albumin_mean']:.1f} vs {group_means.loc['Survived', 'Albumin_mean']:.1f} g/dL). "
    "The mean of each patient's lowest GCS was also lower in the death group "
    f"({group_means.loc['Died', 'GCS_min']:.1f} vs {group_means.loc['Survived', 'GCS_min']:.1f})."))

### 7. Outcome summary

In [ ]:
deaths = wide["In-hospital_death"]
outcome_table = table_data.groupby("ICU")["In-hospital_death"].agg(Stays="size", Deaths="sum")
outcome_table.loc["Overall"] = [len(wide), deaths.sum()]
outcome_table["Mortality (%)"] = 100 * outcome_table.Deaths / outcome_table.Stays
display(outcome_table.round(2))
outcome_table.to_csv(OUT / "outcome_summary.csv")

In [ ]:
icu_mortality = outcome_table.drop(index="Overall")["Mortality (%)"]
highest_icu = icu_mortality.idxmax()
lowest_icu = icu_mortality.idxmin()
display(Markdown(
    f"- There were {outcome_table.loc['Overall', 'Stays']:,.0f} ICU stays and "
    f"{outcome_table.loc['Overall', 'Deaths']:,.0f} deaths. The overall death rate was "
    f"{outcome_table.loc['Overall', 'Mortality (%)']:.2f}%. "
    f"{highest_icu} had the highest death rate ({icu_mortality.loc[highest_icu]:.2f}%). "
    "Surgical ICU and coronary care had similar death rates "
    f"({icu_mortality.loc['Surgical ICU']:.2f}% and {icu_mortality.loc['Coronary care']:.2f}%). "
    f"{lowest_icu} had the lowest death rate ({icu_mortality.loc[lowest_icu]:.2f}%)."))

### 8. Missingness map

Blue means at least one available measurement; gray means none.

In [ ]:
observed = pd.DataFrame({v: wide[f"{v}_count"].gt(0) for v in variables})
missing = (100 * (1 - observed.mean())).sort_values(ascending=False)
display(missing.round(2).rename("Never measured (%)").to_frame())
missing.rename("Never measured (%)").to_csv(OUT / "missingness_summary.csv")
print("Stays with no valid time-series measurements:", int((~observed.any(axis=1)).sum()))
order = observed.sum(axis=1).sort_values().index
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap((~observed.loc[order, missing.index]).T, cmap=["#245778", "#e5e9ed"],
            vmin=0, vmax=1, xticklabels=False, yticklabels=True, cbar=False, ax=ax)
ax.set(title="Blue = observed; gray = never measured", xlabel="ICU stays sorted by coverage", ylabel="")
fig.tight_layout()
fig.savefig(OUT / "missingness.png", dpi=160)
display(Image(filename=str(OUT / "missingness.png"), alt="Measurement availability by variable and ICU stay"))
plt.close(fig)

- More missing data may mean this test is not routine. Doctors may order it when they suspect a problem.
- More complete data may mean this variable is routinely monitored, such as heart rate.

#### Missingness over time

Each cell is the percentage of all stays without a measurement in that hour. The last bin covers hour 47 and includes measurements at 48:00.

In [ ]:
hour = (measurement_minutes // 60).clip(upper=47).rename("Hour")
hourly = measured.groupby([measured.Parameter, hour]).RecordID.nunique().unstack(fill_value=0)
hourly = 100 * (1 - hourly.reindex(index=variables, columns=range(48), fill_value=0) / len(wide))
hourly.to_csv(OUT / "hourly_missingness.csv")
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(hourly, cmap="Blues", vmin=0, vmax=100, xticklabels=4, yticklabels=True,
            cbar_kws={"label": "Stays without a measurement (%)"}, ax=ax)
ax.set(title="Missingness by hour since ICU admission", xlabel="Hour (last bin includes 48:00)", ylabel="")
fig.tight_layout()
fig.savefig(OUT / "missingness_by_hour.png", dpi=160)
display(Image(filename=str(OUT / "missingness_by_hour.png")))
plt.close(fig)


- DiasABP, HR, MAP, SysABP, and Urine: The color becomes lighter in the first few hours and then stays relatively stable. This means more patients have measurements after the first few hours.
  
- Weight: The color is very light at admission, then becomes much darker, and gradually becomes lighter again. Most patients have a weight record at admission, but later records are not available every hour.

- Cholesterol and troponin: The color stays dark during the whole observation period. This means few patients have these measurements in each hour.

### 9. Is missingness informative?

In [ ]:
rows = []
for variable in variables:
    mask = observed[variable]
    rows.append({"Variable": variable, "Measured n": mask.sum(), "Never measured n": (~mask).sum(),
        "Measured deaths": deaths[mask].sum(), "Never measured deaths": deaths[~mask].sum(),
        "Measured mortality (%)": 100 * deaths[mask].mean(),
        "Never measured mortality (%)": 100 * deaths[~mask].mean()})
association = pd.DataFrame(rows).set_index("Variable")
association.to_csv(OUT / "missingness_and_mortality.csv")
display(association.round(2))
lactate = association.loc["Lactate"]

In [ ]:
display(Markdown(
    "- For some variables, the two groups had different death rates. For lactate, the death rate was "
    f"{association.loc['Lactate', 'Measured mortality (%)']:.2f}% in the measured group and "
    f"{association.loc['Lactate', 'Never measured mortality (%)']:.2f}% in the group with no available measurements. "
    f"For Troponin I, the rates were {association.loc['TroponinI', 'Measured mortality (%)']:.2f}% and "
    f"{association.loc['TroponinI', 'Never measured mortality (%)']:.2f}%. "
    f"For Troponin T, the rates were {association.loc['TroponinT', 'Measured mortality (%)']:.2f}% and "
    f"{association.loc['TroponinT', 'Never measured mortality (%)']:.2f}%.\n"
    "- These results suggest that whether a test is recorded may provide information about death risk. "
    "Doctors may order a test when they suspect a problem. Therefore, missingness in some variables "
    "may be useful for prediction."))

### AI use statement

**AI-use statement:** In this assignment, I used OpenAI to understand the assignment requirements, set up the Pixi environment, write Python code, and help to understand the missingness matrix plot.

AI can help me understand this project, but it cannot build an efficient framework. It also adds many unnecessary data checks. These checks make the code more complicated but provide little useful information.